# 예제 05. 전이학습 vs 직접 만든 CNN
빅데이터프로그래밍 · 10주차

## 목표
- 같은 데이터로 두 방법을 비교한다
- 학습 시간과 정확도를 함께 본다
- 세 가지 전이학습 방식도 비교한다

과제와 같은 형식입니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import pandas as pd
import time

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 — 예제 04와 같습니다


In [ ]:
IMG_SIZE = 224
NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), NORM,
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), NORM,
])

raw_train = datasets.Flowers102("./data", split="train", download=True, transform=train_tf)
raw_val   = datasets.Flowers102("./data", split="val",   download=True, transform=eval_tf)

KEEP = [0, 1, 2, 3, 4]
NAMES = ["pink primrose", "hard-leaved orchid", "canterbury bells", "sweet pea", "english marigold"]

class Subset5(torch.utils.data.Dataset):
    def __init__(self, base):
        self.base = base
        self.idx = [i for i, lb in enumerate(base._labels) if lb in KEEP]
        self.remap = {c: i for i, c in enumerate(KEEP)}
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[self.idx[i]]
        return x, self.remap[y]

train_loader = DataLoader(Subset5(raw_train), batch_size=16, shuffle=True)
val_loader   = DataLoader(Subset5(raw_val),   batch_size=32, shuffle=False)
N_CLASSES = len(KEEP)
print(f"학습 {len(train_loader.dataset)}장 · 검증 {len(val_loader.dataset)}장")


## 2. 직접 만든 CNN — 8주차 구조를 224×224에 맞춤


In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


print("직접 만든 CNN 파라미터:", f"{sum(p.numel() for p in ScratchCNN().parameters()):,}개")


## 3. 세 가지 전이학습 방식


In [ ]:
def transfer(mode):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if mode == "fc_only":
        for p in m.parameters():
            p.requires_grad = False
    elif mode == "layer4":
        for p in m.parameters():
            p.requires_grad = False
        for p in m.layer4.parameters():
            p.requires_grad = True
    m.fc = nn.Linear(m.fc.in_features, N_CLASSES)
    return m


## 4. 공통 학습 함수


In [ ]:
loss_fn = nn.CrossEntropyLoss()
EPOCHS = 10

def evaluate(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def run(name, model, lr):
    torch.manual_seed(42)
    model = model.to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    print(f"\n{name}")
    start = time.time()
    hist = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append(evaluate(model, val_loader))
        if epoch % 5 == 0 or epoch == 1:
            print(f"  epoch {epoch:2d}  검증 {hist[-1][1]:.4f}")
    elapsed = time.time() - start
    train_n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {elapsed:.1f}초")
    return {"hist": hist, "time": elapsed, "train_params": train_n, "model": model}


## 5. 네 가지 학습


In [ ]:
results = {}
results["직접 만든 CNN"]    = run("직접 만든 CNN", ScratchCNN(), 1e-3)
results["전이 · fc만"]      = run("전이학습 fc만", transfer("fc_only"), 1e-3)
results["전이 · layer4+fc"] = run("전이학습 layer4+fc", transfer("layer4"), 1e-4)
results["전이 · 전체"]      = run("전이학습 전체 미세조정", transfer("full"), 1e-4)


## 6. 비교표


In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "모델": name,
        "학습 파라미터": f"{r['train_params']:,}",
        "최종 검증 정확도": round(r["hist"][-1][1], 4),
        "최고 검증 정확도": round(max(h[1] for h in r["hist"]), 4),
        "검증 손실": round(r["hist"][-1][0], 4),
        "학습 시간(초)": round(r["time"], 1),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

best = table.loc[table["최고 검증 정확도"].idxmax(), "모델"]
print(f"\n가장 좋은 모델: {best}")


## 7. 학습 곡선 비교


In [ ]:
xs = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for name, r in results.items():
    ax[0].plot(xs, [h[0] for h in r["hist"]], label=name)
    ax[1].plot(xs, [h[1] for h in r["hist"]], label=name)
ax[0].set_title("검증 손실"); ax[1].set_title("검증 정확도")
for a in ax:
    a.set_xlabel("epoch"); a.legend(fontsize=9); a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 8. 시간 대비 정확도
전이학습은 첫 epoch부터 이미 높은 정확도로 시작합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.4))
for name, r in results.items():
    per_epoch = r["time"] / EPOCHS
    ax.plot([per_epoch * (i+1) for i in range(EPOCHS)],
            [h[1] for h in r["hist"]], marker="o", markersize=3, label=name)
ax.set_xlabel("누적 학습 시간(초)"); ax.set_ylabel("검증 정확도")
ax.legend(fontsize=9); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("첫 epoch 검증 정확도")
for name, r in results.items():
    print(f"  {name:18s} {r['hist'][0][1]:.4f}")


## 직접 해보기
1. epoch을 30으로 늘리면 직접 만든 CNN이 전이학습을 따라잡나요?
2. 학습 데이터를 절반으로 줄이면 두 방법의 차이가 어떻게 되나요?
3. `resnet50` 으로 바꾸면 시간과 정확도가 어떻게 변하나요?


In [ ]:
# 여기에 작성하세요
